# Snowflake Account & Organization Management

## Organization

An **organization** is a first-class Snowflake object that links the accounts owned by your business entity.

An organization can consist of the following types of accounts:
- Organization account
- Regular Snowflake account
- Snowflake Open Catalog account

### Organization Creation

Snowflake customers never directly create an organization. For users who sign up for a Snowflake account using the self-service option, an organization is automatically created with a system-generated name when the account is created.

---

## Organization Account

The organization account is a special type of account that organization administrators use to perform tasks that affect the entire organization.

- The `ORGADMIN` role is required to **execute** the `CREATE ORGANIZATION ACCOUNT` command.
- The `GLOBALORGADMIN` role is granted to the initial admin user **inside** the newly created organization account.

```sql
-- Must be executed from an ORGADMIN-enabled account
USE ROLE ORGADMIN;

CREATE ORGANIZATION ACCOUNT myorgaccount
  ADMIN_NAME = admin
  ADMIN_PASSWORD = 'TestPassword1'
  EMAIL = 'myemail@myorg.org'
  MUST_CHANGE_PASSWORD = true
  EDITION = enterprise;
```

> **Note:** Account and Organization both are just containers. Only `ENTERPRISE` or `BUSINESS_CRITICAL` editions are allowed for organization accounts.

---

## Regular Account

**Q. If an account is just a container, why do we need to specify `ADMIN_NAME` and `ADMIN_PASSWORD` while creating an account?**

**Ans.** The account itself is still a container. Snowflake asks you to create the first administrative user inside that new account at the time the account is created. The SQL command combines both actions:
- Create the account (container)
- Create the first admin user inside it

```sql
CREATE ACCOUNT my_account
  ADMIN_NAME = admin
  ADMIN_PASSWORD = 'TestPassword1'
  EMAIL = 'myemail@myacc.org'
  MUST_CHANGE_PASSWORD = true
  EDITION = enterprise;
```

> **Note:** Regular accounts support `STANDARD`, `ENTERPRISE`, or `BUSINESS_CRITICAL` editions.

---

## Best Practice

Create the Organization Account using ORGADMIN-enabled account and perform all subsequent account management activities through the Organization Account. This provides a centralized and scalable model for managing a multi-account Snowflake environment.

## Snowflake Account & Organization Hierarchy

```
Snowflake (Service Provider)
│
└── Organization (Your Company)
    │
    ├── Organization Account (ONLY ONE per org)
    │   │
    │   ├── Initial Admin User
    │   │     └── GLOBALORGADMIN Role Assigned
    │   │
    │   ├── Organization Users
    │   └── Organization User Groups
    │
    ├── DEV Account
    │   │
    │   ├── Initial Admin User
    │   │     └── ACCOUNTADMIN Role Assigned
    │   │
    │   ├── Users
    │   ├── Roles
    │   │     ├── ACCOUNTADMIN
    │   │     ├── SECURITYADMIN
    │   │     ├── SYSADMIN
    │   │     └── Custom Roles
    │   │
    │   ├── Warehouses
    │   │
    │   ├── Databases
    │   │   └── Schemas
    │   │       ├── Tables
    │   │       ├── Views
    │   │       ├── Materialized Views
    │   │       ├── Dynamic Tables
    │   │       ├── Streams
    │   │       ├── Tasks
    │   │       ├── Stages
    │   │       ├── File Formats
    │   │       ├── Functions (UDFs)
    │   │       ├── Procedures
    │   │       └── Sequences
    │   │
    │   ├── Resource Monitors
    │   ├── Network Policies
    │   ├── Shares
    │   └── Other Account-Level Objects
    │
    ├── QA Account
    │   └── (Same structure as DEV)
    │
    ├── PROD Account
    │   └── (Same structure as DEV)
    │
    └── Managed Accounts / Reader Accounts
        ├── Reader Users
        └── Shared Data (read-only consumers)
```

> **Note on `IS_ORG_ADMIN`:** This is an **account-level** property, not a user-level flag. It is set via `ALTER ACCOUNT <name> SET IS_ORG_ADMIN = TRUE` to enable the `ORGADMIN` role in a regular account. However, `ORGADMIN` is being phased out in favor of `GLOBALORGADMIN` in the organization account.

## Snowflake Object Levels Explained

---

### Level 1: Organization

An Organization links all Snowflake accounts owned by the same business entity. It is the top-level administrative construct.

**Facts:**
- Only one organization per business entity
- Organization is NOT a user
- Organization is NOT an account
- You never directly log into an organization

**Think of Organization as:** Company Boundary

**Examples:** MICROSOFT, AMAZON, ADOBE, CONTOSO

---

### Level 2: Organization Account

A special account used to perform tasks affecting the entire organization. It is **not** created by default — you need to create one explicitly. Only one Organization Account can exist per Organization.

**Responsibilities:**
- Regular account lifecycle management
- Manage organization users
- Manage organization user groups
- Organization-wide monitoring
- Organization-wide governance
- Cross-account administration

---

### Level 3: Organization Users

Organization users are special identities intended to work across multiple accounts in an organization. These users are created only in the Organization Account.

---

### Level 4: Regular Accounts

When you sign up for Snowflake for the first time, Snowflake creates one regular account for you and sets `IS_ORG_ADMIN` to True. These are the accounts where actual work happens. Each account is an isolated Snowflake environment.

```
Organization
│
└── Regular Account [Created by Snowflake on sign-up]
     └── Admin User [The sign-up user becomes the Admin]
```

---

### Level 5: Regular Account Users

A user is the actual identity that can log in, run SQL, create tables, and query data (depending on permissions). These users are associated only with a single regular account. They manage and create objects as per their assigned roles.

**Hierarchy inside a regular account:**
```
Account
│
├── Users
├── Roles
├── Warehouses
├── Databases
├── Resource Monitors
├── Shares
└── Policies
```

**Database hierarchy inside an account:**
```
Database
│
└── Schema
    ├── Tables
    ├── Views
    ├── Streams
    ├── Tasks
    ├── Functions
    └── Procedures
```

**User → Role → Privileges relationship:**
```
User
  ↓
Role
  ↓
Privileges
```

Snowflake follows **Role-Based Access Control (RBAC)**. A role is created and assigned privileges, then the role is assigned to a user. In Snowflake, you don't assign any privilege directly to a user.

---

### Level 6: Managed Account (Reader Account)

A managed account is created by data providers to allow consumers to access shared data **without owning a Snowflake account**.

**Commonly used for:**
- Snowflake Data Sharing
- Reader Accounts (read-only consumers)

---

## Key Q&A

**Q. What happens when you sign up for the first time?**

Once you sign up, Snowflake provisions:
- Organization = Created (auto-generated name)
- Account = Created
- Admin User = Created (assigned `ACCOUNTADMIN` role)

```
Organization
│
└── Account
     └── Admin User (ACCOUNTADMIN)
```

You receive: Account URL, Username, Password.

> **Note on `IS_ORG_ADMIN`:** The first account in an organization has the `ORGADMIN` role **enabled at the account level** (not as a user flag). This means you can `USE ROLE ORGADMIN` in that account. However, `ORGADMIN` is being phased out in favor of `GLOBALORGADMIN` in the organization account.

---

**Q. Why does `CREATE ACCOUNT` require a username & password?**

Snowflake performs two actions:
1. Create the account (container)
2. Create the initial admin user (granted `ACCOUNTADMIN` role)

---

**Q. Why does `CREATE ORGANIZATION ACCOUNT` require a username & password?**

Snowflake performs two actions:
1. Create the organization account (container)
2. Create the initial admin user (granted `GLOBALORGADMIN` role)

---

## Important Limitations

| Rule | Detail |
|------|--------|
| Organization Accounts | Only **one** allowed per organization |
| Regular Accounts | Multiple allowed |
| Regular Users | Scoped to **one account** only |
| Organization Users | Can work across **multiple accounts** in the organization |
| Create Regular User | `CREATE USER` |
| Create Organization User | `CREATE ORGANIZATION USER` |

---

**Q. Does each account get a different login URL?**

**Ans.** Yes. Each account has its own unique URL in the format: `https://<orgname>-<accountname>.snowflakecomputing.com`

## Q. If `ORGADMIN` is being phased out, how will someone create a new Organization Account or Regular Account in the future?

**Ans.** The phase-out is about *where* you perform org-level tasks, not about removing the ability to create accounts.

### Current Model (being phased out)

- Sign up → get a regular account with `ORGADMIN` role enabled
- Use `ORGADMIN` from that regular account to create other accounts, the organization account, etc.
- Multiple regular accounts could have `ORGADMIN` enabled (up to 8)

### Future Model (centralized)

- Sign up → get a regular account (still works the same)
- Use `ORGADMIN` to create the Organization Account **once** (bootstrap step)
- After the Organization Account exists, use `GLOBALORGADMIN` inside it for **all** subsequent org-level tasks:
  - Create new regular accounts
  - Create organization users
  - Monitor usage across the org
- `ORGADMIN` in regular accounts becomes unnecessary and will eventually be removed

### Lifecycle

```
1. Sign up         → Regular Account (ORGADMIN enabled)
2. Bootstrap       → CREATE ORGANIZATION ACCOUNT (using ORGADMIN, one-time)
3. Ongoing work    → All done via GLOBALORGADMIN in the Org Account
4. Phase-out       → ORGADMIN disabled in regular accounts (no longer needed)
```

> **Key Point:** You don't need `ORGADMIN` in a regular account to create new regular accounts if you already have an Organization Account. The `GLOBALORGADMIN` role in the org account can do everything `ORGADMIN` could — and more. The phase-out doesn't leave you stranded; it just centralizes the control plane into the Organization Account.

## Q. Once `ORGADMIN` is fully removed, how will a brand-new user create an Organization Account on first sign-up?

**Ans.** The docs state: *"Snowflake will send a notification email to customers at least three months prior to phasing out the ORGADMIN role."*

The most likely outcome (consistent with how Snowflake already auto-creates the organization during sign-up) is that **Snowflake will auto-provision the Organization Account as part of the initial sign-up flow.**

### Expected Future Sign-Up Flow

```
New user signs up
  → Organization created (already happens today)
  → Organization Account created (auto-provisioned, with GLOBALORGADMIN)
  → Regular Account created (with ACCOUNTADMIN)
  → User gets credentials for both
```

This eliminates the need for `ORGADMIN` entirely — there's no manual "bootstrap" step because Snowflake handles org account creation automatically at sign-up.

> **Key Takeaway:** The gap exists only during the transition period. Once Snowflake completes the phase-out, new customers won't need `ORGADMIN` at all because the Organization Account will be created for them automatically.